<a href="https://colab.research.google.com/github/lu40307/pygmt-map-lab/blob/main/%E5%9C%B0%E5%8B%99%E4%BD%9C%E6%A5%AD2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, pandas
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    print("步驟 1/2：安裝 Conda（約 1 分鐘）。完成後 Colab 會自動重啟執行環境，等重新連線再執行下一格。", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.13"])
    import condacolab
    condacolab.install()
else:
    print("本機模式：使用目前 Python 環境。")

步驟 1/2：安裝 Conda（約 1 分鐘）。完成後 Colab 會自動重啟執行環境，等重新連線再執行下一格。

📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

✨🍰✨ Everything looks OK!


In [11]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, pandas
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    print("步驟 2/2：安裝 PyGMT 與相依套件（約 2–4 分鐘），下方會逐行顯示進度。", flush=True)
    command = ["mamba", "install", "-y", "-c", "conda-forge", "pygmt=0.17", "gmt=6.5", "ghostscript=10.04", "pandas"]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
    if process.returncode != 0:
        raise RuntimeError(f"安裝失敗（exit {process.returncode}），請重新執行本格或重啟執行環境。")
    print("安裝完成，可以往下執行。")
else:
    print("跳過 Colab 安裝。")

步驟 2/2：安裝 PyGMT 與相依套件（約 2–4 分鐘），下方會逐行顯示進度。
conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache

Pinned packages:

  - python=3.13

Pinned packages:

  - python_abi[version="=3.13",build="*cp313*"]

Pinned packages:

  - cuda-version*.*

Pinned packages:

  - python=3.13


Transaction

  Prefix: /usr/local

  All requested packages already installed


Transaction starting

Transaction finished

安裝完成，可以往下執行。


In [12]:
!git clone https://github.com/lu40307/pygmt-map-lab.git

fatal: destination path 'pygmt-map-lab' already exists and is not an empty directory.


In [17]:
%run /content/pygmt-map-lab/examples/02_region_map_section.py

USGS 2000-01-01 起 M >= 5.0：2877 筆
走廊內 444 筆；A–B 長 1564 km


grdblend [NOTICE]: Remote data courtesy of GMT data server oceania [http://oceania.generic-mapping-tools.org]
grdblend [NOTICE]: SRTM15 Earth Relief v2.7 at 02x02 arc minutes reduced by Gaussian Cartesian filtering (10.5 km fullwidth) [Tozer et al., 2019].
grdblend [NOTICE]:   -> Download 60x60 degree grid tile (earth_relief_02m_g): N30E120


圖說骨架：USGS 2000-01-01 起 M >= 5.0，範圍 [128, 150, 30, 46]，A=(130, 38.5)、B=(148, 38.5)，走廊全寬 200 km；剖面 VE = 1.3x；走廊內 21% 的深度是 USGS 預設值（10／33 km）。
saved region_map.png region_section.png


In [24]:
import datetime
import pandas as pd
import pygmt

# ==========================================
# 1. 參數與區域設定 (東太平洋中洋脊 EPR 範例)
# ==========================================
region = [-120, -95, -15, 5]  # [西經120, 西經95, 南緯15, 北緯5]
pointA = [-115, -12]  # 剖面 A 點 (經度, 緯度)
pointB = [-100, 2]  # 剖面 B 點 (經度, 緯度)
corridor_width = 150  # 走廊全寬 (km)

start_time = "2000-01-01"
end_time = datetime.datetime.now().strftime("%Y-%m-%d")

# ==========================================
# 2. 自動查詢 USGS 地震資料 (超過 20,000 筆自動提高 M)
# ==========================================
min_magnitude = 5.0
max_events = 20000
df_eq = pd.DataFrame()

print("正在向 USGS 查詢地震資料...")

while True:
    url = (
        f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
        f"&starttime={start_time}&endtime={end_time}"
        f"&minmagnitude={min_magnitude}"
        f"&minlongitude={region[0]}&maxlongitude={region[1]}"
        f"&minlatitude={region[2]}&maxlatitude={region[3]}"
    )
    try:
        temp_df = pd.read_csv(url)
        count = len(temp_df)
        print(f"規模 M >= {min_magnitude:.1f}：搜尋到 {count} 筆地震。")

        if count <= max_events:
            df_eq = temp_df
            break
        else:
            print(
                f"⚠️ 筆數超過 {max_events} 筆限制，調高規模至 M >= {min_magnitude + 0.2:.1f} 重試..."
            )
            min_magnitude += 0.2
    except Exception as e:
        print(f"下載失敗: {e}")
        break

# 整理地震資料 (順序：經度, 緯度, 深度)
eq_data = df_eq[["longitude", "latitude", "depth"]].dropna()

# ==========================================
# 3. 定義三段深度色階 (CPT)
# 0–70 km (紅色), 70–300 km (綠色), 300–700 km (藍色)
# ==========================================
cpt_content = """0	red	70	red
70	green	300	green
300	blue	700	blue
"""
with open("eq_depth.cpt", "w") as f:
    f.write(cpt_content)

# ==========================================
# 4. 繪製平面地圖 (region_map.png)
# ==========================================
fig_map = pygmt.Figure()

# 載入 SRTM15+ 高程網格並畫地形底圖
grid = pygmt.datasets.load_earth_relief(resolution="02m", region=region)
fig_map.grdimage(grid=grid, cmap="geo", frame=True)
fig_map.coast(shorelines="0.5p,black", borders="1/0.5p,gray")

# 畫地震點
fig_map.plot(
    data=eq_data,
    style="c0.12c",
    fill="+z",
    cmap="eq_depth.cpt",
    pen="0.2p,black",
)

# 標示 A-B 剖面線與端點
fig_map.plot(
    x=[pointA[0], pointB[0]],
    y=[pointA[1], pointB[1]],
    pen="2p,red",
)
fig_map.text(
    x=[pointA[0], pointB[0]],
    y=[pointA[1], pointB[1]],
    text=["A", "B"],
    font="14p,Helvetica-Bold,red",
    justify="CM",
)

# 新增比例尺與標題
fig_map.basemap(
    map_scale="jBL+w500k+o0.5c/0.5c+f+l",
    frame=[
        '+t"East Pacific Rise Seismicity"',
        f'+s"Source: USGS (M>={min_magnitude:.1f}, N={len(eq_data)})"',
    ],
)

fig_map.savefig("region_map.png")
print("✅ 已成功儲存平面圖: region_map.png")

# ==========================================
# 5. 計算並繪製 A-B 剖面圖 (region_section.png)
# ==========================================
# 將地震資料投射到 A-B 剖面上 (width 接受 [min_width, max_width] 走廊範圍)
half_w = corridor_width / 2
projected_eq = pygmt.project(
    data=eq_data,
    center=pointA,
    endpoint=pointB,
    width=[-half_w, half_w],
    unit=True,
)

fig_sec = pygmt.Figure()

# 取出沿線距離 (p-col, 索引3) 與 深度 (depth, 索引2)
sec_data = projected_eq.iloc[:, [3, 2, 2]]  # [X(距離), Y(深度), Z(深度上色用)]

profile_length = float(sec_data.iloc[:, 0].max()) if len(sec_data) > 0 else 1000
ve_factor = round((700 / profile_length) * (15 / 10), 2)  # 計算 VE 倍率

# 設定座標軸：X 軸距離 (0~L km)，Y 軸深度 (0~700 km，向下)
fig_sec.basemap(
    region=[0, profile_length, 0, 700],
    projection="X15c/-10c",
    frame=[
        'xaf+l"Distance along profile (km)"',
        'ya100f50+l"Depth (km)"',
        f'WSne+t"Cross Section A-B (VE = {ve_factor}x)"',
    ],
)

# 繪製剖面點
if len(sec_data) > 0:
    fig_sec.plot(
        data=sec_data,
        style="c0.15c",
        fill="+z",
        cmap="eq_depth.cpt",
        pen="0.2p,black",
    )

# 標示資料來源
fig_sec.text(
    x=profile_length * 0.02,
    y=660,
    text=f"Source: USGS | Corridor: {corridor_width}km | Events: {len(sec_data)}",
    font="10p,Helvetica,black",
    justify="ML",
)

fig_sec.savefig("region_section.png")
print("✅ 已成功儲存剖面图: region_section.png")


正在向 USGS 查詢地震資料...
規模 M >= 5.0：搜尋到 220 筆地震。
✅ 已成功儲存平面圖: region_map.png
✅ 已成功儲存剖面图: region_section.png
